# MODULE 2 : **POO Avancée et Abstraction**



## I. **ENCAPSULATION & PROPRIÉTÉS**

Dans ce premier chapitre, nous allons apprendre à protéger les données d'un objet (comme le solde d'un compte bancaire) pour éviter qu'elles ne soient modifiées de manière incohérente depuis l'extérieur.


### 1. **La Visibilité des Attributs en Python : `__` vs `_`**

En Python, la gestion de la visibilité repose sur des symboles placés devant le nom de la variable :

#### **Double underscore `__` (Attribut Privé)** 

Quand une variable commence par `__` (ex: `self.__solde`), Python applique un mécanisme automatique appelé **Name Mangling**.
En arrière-plan, Python renomme la variable en `_NomDeLaClasse__solde`. Cela empêche tout accès direct ou modification accidentelle depuis l'extérieur de la classe :

```python
compte = CompteBancaire("Hulrich", 1000)
print(compte.__solde)  # 🚨 Erreur : AttributeError! L'accès direct est bloqué.

```



> **💡 Information complémentaire : Le simple underscore `_` (Attribut Protégé)**
> Si vous croisez un attribut avec un seul underscore (ex: `self._solde`), il s'agit d'une **simple convention visuelle** entre développeurs.
> Python ne bloque techniquement aucun accès : cela signifie simplement *"Cet attribut est interne à la classe, évitez de y toucher directement"*.
> Dans notre cours, nous utiliserons le **double underscore `__**` pour appliquer une restriction stricte.



### 2. **Contrôler l'Accès avec `@property` et `@attribut.setter`**

Au lieu d'écrire des méthodes lourdes comme `get_solde()` ou `set_solde()`, Python utilise le décorateur natif `@property`. Il permet de lire et modifier un attribut privé avec la syntaxe simple d'une variable (`compte.solde = 500`), tout en exécutant du code de validation en arrière-plan.

```python
class CompteBancaire:
    def __init__(self, titulaire, solde_initial=0.0):
        self.titulaire = titulaire
        self.__solde = solde_initial  # Attribut strictement privé

    # 1. GETTER : Permet de lire le solde (ex: print(compte.solde))
    @property
    def solde(self):
        return self.__solde

    # 2. SETTER : Intercepte et valide toute modification (ex: compte.solde = 500)
    @solde.setter
    def solde(self, nouveau_solde):
        if nouveau_solde < 0:
            raise ValueError("❌ Opération refusée : Le solde ne peut pas être négatif.")
        self.__solde = nouveau_solde

    # 3. Méthode métier
    def deposer(self, montant):
        if montant > 0:
            # On passe par le setter pour modifier la valeur en toute sécurité
            self.solde += montant
            print(f"✅ Dépôt de {montant} $ effectué. Nouveau solde : {self.solde} $")

```

#### **Démonstration d'utilisation** :

```python
compte = CompteBancaire("Hulrich", 500.0)

# Lecture via le getter
print(compte.solde)  # Affiche : 500.0

# Modification valide via le setter
compte.solde = 800.0
print(compte.solde)  # Affiche : 800.0

# Tentative de modification invalide
compte.solde = -100.0  # 🚨 Lève une ValueError

## **EXERCICE D'APPLICATION**

### 🏋️ EXERCICE 1 : **Le Contrôle de Prix** (E-commerce)

**Domaine :** Gestion de catalogue produit.

**Énoncé :**

1. Créez une classe `Produit` dont le constructeur prend un `nom` (public) et un `prix` (privé `__prix`).
2. Créez un **getter** `@property` pour lire la propriété `prix`.
3. Créez un **setter** `@prix.setter` qui intercepte les modifications du prix :
   * Le prix doit être strictement supérieur à `0`.
   * Si le prix proposé est inférieur ou égal à `0`, le setter doit lever une `ValueError("Le prix d'un produit doit être positif.")`.


4. Ajoutez une propriété en lecture seule `prix_avec_tva` (getter uniquement) qui calcule et renvoie le prix avec une TVA de 20% (`self.prix * 1.20`).

---

### 🏋️ EXERCICE 2 : **Le Capteur de Température** (Domotique / IoT)

**Domaine :** Mesure physique et sécurité matérielle.

**Énoncé :**
Un capteur mesure la température en degrés Celsius. En physique, la température ne peut jamais descendre en dessous du zéro absolu ($-273.15\text{ °C}$).

1. Créez une classe `CapteurTemperature` avec un attribut privé `__temperature` (par défaut `0.0`).
2. Écrivez le **getter** `@property` pour lire `temperature`.
3. Écrivez le **setter** `@temperature.setter` :
   * Si la température attribuée est inférieure à `-273.15`, le setter lève une `ValueError("Température sous le zéro absolu impossible !")`.
   * Sinon, la valeur est mise à jour.


4. Ajoutez une propriété en lecture seule `temperature_fahrenheit` qui convertit et renvoie la température actuelle en Fahrenheit avec la formule : $F = C \times 1.8 + 32$.

---

### 🏋️ EXERCICE 3 : **La Jauge de Carburant** (Logistique)

**Domaine :** Transport et réservoir de véhicule.

**Énoncé :**
On souhaite gérer le niveau de carburant d'un camion sans risquer le débordement du réservoir.

1. Créez une classe `Camion` qui prend en paramètre `capacite_max` (ex: 100 litres) et initialise `__niveau_essence` à `0.0`.
2. Créez le **getter** `@property` pour `niveau_essence`.
3. Créez le **setter** `@niveau_essence.setter` pour valider les ajouts de carburant :
   * Le niveau ne peut pas être négatif.
   * Le niveau ne peut pas dépasser `capacite_max`.
   * Si une valeur incorrecte est fournie, le setter lève une `ValueError("Niveau de carburant hors des limites du réservoir.")`.


4. Ajoutez une méthode `ajouter_carburant(litres)` qui utilise le setter pour incrémenter le niveau.

---

### 🏋️ EXERCICE 4 : **L'Âge de l'Utilisateur** (Profil & Réseaux Sociaux)

**Domaine :** Validation de compte utilisateur.

**Énoncé :**
Sur une plateforme en ligne, la création de compte exige que l'utilisateur soit majeur ($\ge 18$ ans) et l'âge ne peut techniquement pas dépasser 120 ans.

1. Créez une classe `ProfilUtilisateur` qui prend `pseudo` et `age`.
2. L'attribut `__age` doit être strictement privé.
3. Écrivez le **getter** `@property` pour `age`.
4. Écrivez le **setter** `@age.setter` :
   * Si l'âge attribué est inférieur à `18` ou supérieur à `120`, le setter lève une `ValueError("L'âge doit être compris entre 18 et 120 ans.")`.


5. Dans le constructeur `__init__`, veillez à assigner l'âge en passant directement par le setter (`self.age = age`) afin que la règle de validation s'applique dès l'instanciation de l'objet !

---

## II. **DÉCOUVRIR LE MODULE `abc` & LES CLASSES ABSTRAITES**

En Python, la programmation orientée objet est très flexible. Mais dans de grands projets, cette flexibilité peut devenir un problème : comment forcer différentes classes à suivre la même structure et à posséder les mêmes méthodes ?

C'est là qu'interviennent les **Classes Abstraites** (*Abstract Base Classes* ou **ABC**).



### 1. **À quoi sert une Classe Abstraite ?**

Une classe abstraite est un **contrat ou un plan de construction**. Elle permet de définir un modèle commun pour toute une famille d'objets, avec deux garanties fondamentales :

1. **Impossible d'instancier la classe mère :** On ne peut pas créer un objet directement à partir d'une classe abstraite. Elle sert uniquement de base pour d'autres classes.
2. **Obligation d'implémentation :** Les classes filles qui en héritent sont **obligées** de rédiger le code des méthodes dites *abstraites*. Si elles oublient de le faire, Python refuse de créer l'objet.


### 2. **La Syntaxe du Module Natif `abc`**

Pour créer une classe abstraite en Python, on utilise deux éléments du module `abc` :

* **`ABC`** : La classe de base dont notre classe mère doit hériter.
* **`@abstractmethod`** : Le décorateur qui marque les méthodes que chaque classe fille aura l'obligation d'écrire.

```python
from abc import ABC, abstractmethod

# 1. On hérite de ABC pour en faire une classe abstraite
class AppareilElectrique(ABC):
    def __init__(self, marque):
        self.marque = marque

    # 2. Méthode classique (commune à tous les appareils)
    def afficher_marque(self):
        print(f"Marque : {self.marque}")

    # 3. Méthode abstraite : LE CONTRAT (pas de code ici)
    @abstractmethod
    def allumer(self):
        """Chaque appareil doit définir SA façon de s'allumer."""
        pass
```

#### **Regardons les règles de sécurité en action** :

```python
# 🚨 TENTATIVE 1 : Instancier la classe mère
appareil = AppareilElectrique("Samsung")
# ❌ ERREUR : TypeError: Can't instantiate abstract class AppareilElectrique with abstract method allumer


# 🚨 TENTATIVE 2 : Créer une classe fille qui "oublie" d'écrire 'allumer'
class Radio(AppareilElectrique):
    pass

# radio = Radio("Sony")
# ❌ ERREUR : Python bloque l'instanciation tant que 'allumer' n'est pas définie !


# ✅ CAS VALIDE : La classe fille respecte le contrat
class Television(AppareilElectrique):
    def allumer(self):
        print("📺 L'écran s'illumine et l'image apparaît.")

tv = Television("LG")
tv.allumer()  # Fonctionne parfaitement !

```

## EXERCICES D'APPLICATION


### 🏋️ Exercice 1 : **Le Système de Paiement**

**Énoncé :**
On souhaite créer un module de paiement pour une application mobile.

1. Importez `ABC` et `abstractmethod` depuis le module `abc`.
2. Créez une classe abstraite nommée `MoyenDePaiement(ABC)` :
   * Le constructeur `__init__` prend un argument `montant`.
   * Créez une méthode abstraite `@abstractmethod` nommée `payer()`.


3. Créez une classe fille `PaiementCarte` qui hérite de `MoyenDePaiement` :
   * Son constructeur prend `montant` et `numero_carte`.
   * Implémentez la méthode `payer()` qui affiche : `f"💳 Paiement de {self.montant} $ effectué avec la carte {self.numero_carte}."`


4. Créez une deuxième classe fille `PaiementMobile` qui hérite de `MoyenDePaiement` :
   * Son constructeur prend `montant` et `numero_telephone`.
   * Implémentez la méthode `payer()` qui affiche : `f"📱 Transfert de {self.montant} $ envoyé au numéro {self.numero_telephone}."`



**À faire :**

* Écrire le code des trois classes.
* Instancier un `PaiementCarte` et un `PaiementMobile`, puis appeler leur méthode `payer()`.

Voici 3 nouveaux exercices complémentaires pour bien ancrer le fonctionnement de `ABC` et `@abstractmethod` dans des domaines complètement différents.

---

### 🏋️ EXERCICE 2 : **Le Système d'Notifications** (Services Web)

**Domaine :** Architecture système et communication client.

**Énoncé :**
Dans une plateforme web, on doit pouvoir envoyer des notifications par différents canaux (Email, SMS, Push) sans réécrire la logique d'envoi à chaque fois.

1. Créez une classe abstraite `Notification(ABC)` :
   * Le constructeur prend le `destinataire` et le `message`.
   * Définissez une méthode abstraite `@abstractmethod` nommée `envoyer()`.


2. Créez la classe fille `NotificationEmail` :
   * Son constructeur prend `destinataire`, `message` et `sujet`.
   * Implémentez la méthode `envoyer()` qui affiche : `f"📧 [Email - {self.sujet}] Envoyé à {self.destinataire} : {self.message}"`.


3. Créez la classe fille `NotificationSMS` :
   * Son constructeur prend `destinataire` et `message`.
   * Implémentez la méthode `envoyer()` qui affiche : `f"💬 [SMS] Envoyé au {self.destinataire} : {self.message}"`.



---

### 🏋️ EXERCICE 3 : **La Pipeline de Traitement de Données** (Data Processing)

**Domaine :** Analyse et préparation de données.

**Énoncé :**
Dans un projet Data, on reçoit souvent des chaînes de caractères brutes qu'il faut nettoyer ou transformer avant analyse. On crée une classe abstraite `NettoyeurTexte(ABC)` qui impose une méthode de transformation `nettoyer(texte)`.

1. Créez une classe abstraite `NettoyeurTexte(ABC)` :
   * Elle ne nécessite pas de constructeur `__init__`.
   * Définissez une méthode abstraite `@abstractmethod` nommée `nettoyer(self, texte)`.


2. Créez la classe fille `NettoyeurEspaces` :
   * Implémentez la méthode `nettoyer(self, texte)` pour qu'elle supprime les espaces inutiles au début et à la fin du texte (`.strip()`) et renvoie le texte nettoyé.


3. Créez la classe fille `NettoyeurCasse` :
   * Implémentez la méthode `nettoyer(self, texte)` pour qu'elle passe tout le texte en minuscules (`.lower()`) et renvoie le texte transformé.



---

### 🏋️ EXERCICE 4 : **La Gestion des Employés et de la Paie** (RH)

**Domaine :** Ressources Humaines et calcul de salaire.

**Énoncé :**
Dans une entreprise, chaque employé a une façon différente d'être rémunéré (salarié au mois vs à l'heure), mais le système RH doit pouvoir exécuter la méthode `calculer_salaire()` de manière uniforme.

1. Créez une classe abstraite `Employe(ABC)` :
   * Le constructeur prend `nom` et `identifiant`.
   * Définissez une méthode abstraite `@abstractmethod` nommée `calculer_salaire()`.


2. Créez la classe fille `EmployeMensuel` :
   * Son constructeur prend `nom`, `identifiant` et `salaire_mensuel_fixe`.
   * Implémentez `calculer_salaire()` qui renvoie simplement `salaire_mensuel_fixe`.


3. Créez la classe fille `EmployeHoraire` :
   * Son constructeur prend `nom`, `identifiant`, `taux_horaire` et `heures_travaillees`.
   * Implémentez `calculer_salaire()` qui renvoie `taux_horaire * heures_travaillees`.